# Universidad San Ignacio de Loyola (USIL) — Facultad de Ingeniería
## Notebook 01: Pipeline ETL, Limpieza, Homologación y Perfilado Global (ILOSTAT 2010–2026)

**Responsabilidad Única (Principio SRP):** Ingestar las 9 series oficiales crudas de `data/raw/`, homologar sus dimensiones y generar los datasets maestros inmutables `ilostat_latam_2010_2026_master_long.csv` (23 891 registros) e `ilostat_latam_2010_2026_master_wide.csv` (7 518 filas) en `data/processed/`.

> **Nota de arquitectura:** La delimitación temporal del artículo (**2015–2025**, 20 países sin `X21`, **15 305 observaciones**) y la auditoría numérica cruzada contra las medidas DAX de `powerbi/Dashboard_V2.pbix` se ejecutan de forma independiente (solo lectura) en [`02_Auditoria_y_Validacion_2015_2025_DAX.ipynb`](02_Auditoria_y_Validacion_2015_2025_DAX.ipynb).

## Configuración Inicial del Entorno y Librerías

In [1]:
import os
import glob
import pandas as pd
import numpy as np

# Configuración visual de pandas para visualización académica
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)
pd.set_option('display.precision', 2)

print("Entorno configurado correctamente.")
print(f"Versión de Pandas: {pd.__version__}")


Entorno configurado correctamente.
Versión de Pandas: 3.1.1


---
## Etapa 1: Ingesta Automatizada desde la API de ILOSTAT (`data/raw/`)

Los datos se descargaron mediante el endpoint oficial de descarga masiva de ILOSTAT:
$$\text{URL} = \text{https://rplumber.ilo.org/data/indicator/?id}=\{ID\}\&\text{type=both}\&\text{format=.csv}$$

Se filtraron exclusivamente los **20 países de América Latina** más el **agregado regional de América Latina y el Caribe (código X21)** para el periodo **2010–2026**.


In [2]:
raw_folder = os.path.join("..", "data", "raw")
csv_files = sorted(glob.glob(os.path.join(raw_folder, "*.csv")))

print(f"Total de archivos CSV originales encontrados en 'data/raw/': {len(csv_files)}\n")
raw_summary = []
for f in csv_files:
    fname = os.path.basename(f)
    df_temp = pd.read_csv(f, low_memory=False)
    raw_summary.append({
        "Archivo": fname,
        "Indicador OIT": df_temp["indicator"].iloc[0] if "indicator" in df_temp.columns else "N/A",
        "Total Registros": len(df_temp),
        "Columnas": len(df_temp.columns),
        "Año Mínimo": int(df_temp["time"].min()) if "time" in df_temp.columns else "N/A",
        "Año Máximo": int(df_temp["time"].max()) if "time" in df_temp.columns else "N/A",
    })

df_raw_summary = pd.DataFrame(raw_summary)
display(df_raw_summary)


Total de archivos CSV originales encontrados en 'data/raw/': 9



,Archivo,Indicador OIT,Total Registros,Columnas,Año Mínimo,Año Máximo
0,01_tasa_participacion_modelada_EAP_2WAP_SEX_AG...,EAP_2WAP_SEX_AGE_RT,3213,14,2010,2026
1,02_relacion_empleo_poblacion_modelada_EMP_2WAP...,EMP_2WAP_SEX_AGE_RT,3213,12,2010,2026
2,03_tasa_desempleo_modelada_UNE_2EAP_SEX_AGE_RT...,UNE_2EAP_SEX_AGE_RT,3213,14,2010,2026
3,04_tasa_neet_jovenes_modelada_EIP_2EET_SEX_RT.csv,EIP_2EET_SEX_RT,1071,12,2010,2026
4,05a_tasa_informalidad_modelada_sexo_EMP_2IFL_S...,EMP_2IFL_SEX_RT,51,10,2010,2026
5,05b_tasa_informalidad_nacional_edad_EMP_NIFL_S...,EMP_NIFL_SEX_AGE_RT,2124,20,2010,2025
6,06_tasa_trabajadores_pobres_modelada_SDG_0111_...,SDG_0111_SEX_AGE_RT,2560,14,2010,2025
7,07_desempleo_urbano_rural_UNE_DEAP_SEX_AGE_GEO...,UNE_DEAP_SEX_AGE_GEO_RT,6471,22,2010,2025
8,08_neet_urbano_rural_EIP_NEET_SEX_GEO_RT.csv,EIP_NEET_SEX_GEO_RT,1989,18,2010,2025


---
## Etapa 2: Estandarización, Homologación de Dimensiones y Limpieza

Para garantizar la interoperabilidad con la revista internacional y Power BI:
1. **Países:** Mapeo de códigos ISO-3 a nombres oficiales en inglés (`PER` $\rightarrow$ `Peru`, `BRA` $\rightarrow$ `Brazil`, etc.).
2. **Sexo:** Homologación de códigos (`SEX_T` $\rightarrow$ `Total`, `SEX_F` $\rightarrow$ `Female`, `SEX_M` $\rightarrow$ `Male`).
3. **Edad:** Estandarización a tres grupos (`Youth (15-24)`, `Adults (25+)`, `Total (15+)`). En indicadores NEET, la población es estrictamente `Youth (15-24)`.
4. **Área Geográfica:** Distinción de series nacionales (`National`) frente a encuestas desagregadas (`Urban`, `Rural`).
5. **Control de Calidad:** Verificación de rango porcentual ($0\% \le x \le 100\%$) y eliminación de duplicados dimensionales.


In [3]:
# Diccionarios maestros de homologación
COUNTRY_MAP = {
    "ARG": "Argentina", "BOL": "Bolivia", "BRA": "Brazil", "CHL": "Chile",
    "COL": "Colombia", "CRI": "Costa Rica", "CUB": "Cuba", "DOM": "Dominican Republic",
    "ECU": "Ecuador", "SLV": "El Salvador", "GTM": "Guatemala", "HTI": "Haiti",
    "HND": "Honduras", "MEX": "Mexico", "NIC": "Nicaragua", "PAN": "Panama",
    "PRY": "Paraguay", "PER": "Peru", "URY": "Uruguay", "VEN": "Venezuela",
    "X21": "Latin America and the Caribbean (Aggregate)"
}

SEX_MAP = {
    "SEX_T": "Total",
    "SEX_M": "Male",
    "SEX_F": "Female"
}

AGE_MAP = {
    "AGE_YTHADULT_Y15-24": "Youth (15-24)",
    "AGE_YTHADULT_YGE25": "Adults (25+)",
    "AGE_YTHADULT_YGE15": "Total (15+)"
}

GEO_MAP = {
    "GEO_COV_NAT": "National",
    "GEO_COV_URB": "Urban",
    "GEO_COV_RUR": "Rural"
}

INDICATOR_NAMES = {
    "EAP_2WAP_SEX_AGE_RT": "LFPR_Modelled",          # Tasa de Participación Laboral (%)
    "EMP_2WAP_SEX_AGE_RT": "EPR_Modelled",           # Relación Empleo-Población (%)
    "UNE_2EAP_SEX_AGE_RT": "UR_Modelled",            # Tasa de Desempleo (%)
    "EIP_2EET_SEX_RT": "NEET_Youth_Modelled",        # Tasa NEET Juvenil 15-24 (%)
    "EMP_2IFL_SEX_RT": "Informality_Modelled_Total", # Informalidad Modelada por Sexo (%)
    "EMP_NIFL_SEX_AGE_RT": "Informality_National",   # Informalidad Nacional por Edad (%)
    "EAR_2WPR_SEX_AGE_RT": "Working_Poverty_Modelled", # Trabajadores Pobres (ODS 1.1.1)
    "SDG_0111_SEX_AGE_RT": "Working_Poverty_Modelled", # Código OIT ODS 1.1.1
    "UNE_DEAP_SEX_AGE_GEO_RT": "UR_Urban_Rural",     # Desempleo Urbano/Rural (%)
    "EIP_NEET_SEX_GEO_RT": "NEET_Urban_Rural"        # NEET Urbano/Rural (%)
}

print("Diccionarios de estandarización inicializados.")


Diccionarios de estandarización inicializados.


---
## Etapa 3: Unificación en un solo Dataset Maestro (`data/processed/`)

Se construyen dos estructuras complementarias:
1. **Formato Largo (*Long / Tidy Data*):** Ideal para filtrado dinámico, segmentadores multidimensionales y cálculo de relaciones en Power BI.
2. **Formato Ancho (*Wide / Pivoted Matrix*):** Tabla consolidada donde cada indicador es una columna numérica, permitiendo calcular ratios y brechas directamente entre columnas.


In [4]:
# Procesamiento y unificación
proc_folder = os.path.join("..", "data", "processed")
os.makedirs(proc_folder, exist_ok=True)

# Evitar duplicar archivos si existen aliases
unique_csv_files = []
for f in csv_files:
    fname = os.path.basename(f)
    if "EAR_2WPR" in fname and any("SDG_0111" in os.path.basename(x) for x in csv_files):
        continue
    unique_csv_files.append(f)

unified_records = []
for file in unique_csv_files:
    df_curr = pd.read_csv(file, low_memory=False)
    if df_curr.empty:
        continue
    for _, row in df_curr.iterrows():
        ind_code = str(row.get("indicator", ""))
        c_1 = str(row.get("classif1", ""))
        c_2 = str(row.get("classif2", ""))
        
        # Asignación de grupo de edad
        if ind_code in ["EIP_2EET_SEX_RT", "EIP_NEET_SEX_GEO_RT"]:
            age_group = "Youth (15-24)"
        else:
            age_group = AGE_MAP.get(c_1, "Total (15+)")
            
        # Asignación de área geográfica
        if "GEO_" in c_1:
            area_type = GEO_MAP.get(c_1, "National")
        elif "GEO_" in c_2:
            area_type = GEO_MAP.get(c_2, "National")
        else:
            area_type = "National"
            
        val = row.get("obs_value")
        try:
            val_pct = round(float(val), 2) if pd.notnull(val) else None
        except:
            val_pct = None
            
        iso = str(row.get("ref_area", ""))
        unified_records.append({
            "ISO3": iso,
            "Country": COUNTRY_MAP.get(iso, iso),
            "Is_Regional_Aggregate": True if iso == "X21" else False,
            "Year": int(row.get("time")) if pd.notnull(row.get("time")) else None,
            "Sex_Code": row.get("sex"),
            "Sex": SEX_MAP.get(row.get("sex"), "Other"),
            "Age_Group": age_group,
            "Area_Type": area_type,
            "Indicator_ID": ind_code,
            "Indicator_Short": INDICATOR_NAMES.get(ind_code, ind_code),
            "Value_Pct": val_pct
        })

df_long = pd.DataFrame(unified_records)
df_long = df_long.dropna(subset=["Value_Pct", "Year"])
df_long["Year"] = df_long["Year"].astype(int)

# Filtro estricto de sexos estándar y eliminación de duplicados
filter_sex = df_long["Sex"].isin(["Total", "Male", "Female"])
df_long = df_long[filter_sex].drop_duplicates(
    subset=["ISO3", "Year", "Sex", "Age_Group", "Area_Type", "Indicator_Short"]
)
df_long = df_long.sort_values(["Country", "Year", "Sex", "Age_Group", "Area_Type", "Indicator_Short"])

long_csv_path = os.path.join(proc_folder, "ilostat_latam_2010_2026_master_long.csv")
df_long.to_csv(long_csv_path, index=False, encoding="utf-8-sig")

# Generación de Formato Ancho (Wide)
df_wide = df_long.pivot_table(
    index=["ISO3", "Country", "Is_Regional_Aggregate", "Year", "Sex", "Age_Group", "Area_Type"],
    columns="Indicator_Short",
    values="Value_Pct",
    aggfunc="first"
).reset_index()
df_wide = df_wide.sort_values(["Country", "Year", "Sex", "Age_Group", "Area_Type"])

wide_csv_path = os.path.join(proc_folder, "ilostat_latam_2010_2026_master_wide.csv")
df_wide.to_csv(wide_csv_path, index=False, encoding="utf-8-sig")

print(f"Dataset Maestro Largo guardado: {long_csv_path} ({len(df_long):,} registros)")
print(f"Dataset Maestro Ancho guardado: {wide_csv_path} ({len(df_wide):,} filas x {len(df_wide.columns)} columnas)")


Dataset Maestro Largo guardado: ..\data\processed\ilostat_latam_2010_2026_master_long.csv (23,891 registros)
Dataset Maestro Ancho guardado: ..\data\processed\ilostat_latam_2010_2026_master_wide.csv (7,518 filas x 16 columnas)


---
## Etapa 5: Análisis Exploratorio (EDA) y los 8 Hallazgos para la Sección de Resultados

En esta sección se calculan las estadísticas descriptivas y se extraen los valores empíricos exactos para redactar los **dos hallazgos analíticos por cada uno de los cuatro dashboards** exigidos en la rúbrica del paper.


In [5]:
# Estadísticas Descriptivas Globales por Indicador
print("=== Resumen Estadístico Descriptivo de Indicadores Clave en América Latina (2010 - 2026) ===")
summary_stats = df_long.groupby("Indicator_Short")["Value_Pct"].agg(
    ["count", "mean", "std", "min", lambda x: np.percentile(x, 25), "median", lambda x: np.percentile(x, 75), "max"]
).rename(columns={"<lambda_0>": "p25", "<lambda_1>": "p75"})

summary_stats = summary_stats.round(2)
display(summary_stats)


=== Resumen Estadístico Descriptivo de Indicadores Clave en América Latina (2010 - 2026) ===


,count,mean,std,min,p25,median,p75,max
Indicator_Short,,,,,,,,
EPR_Modelled,3213,55.08,16.62,14.75,43.07,54.97,67.40,89.97
Informality_Modelled_Total,51,34.53,1.48,31.71,32.98,34.68,35.98,36.80
Informality_National,2124,59.71,18.11,18.82,45.06,60.82,73.45,98.02
LFPR_Modelled,3213,59.70,15.65,18.06,48.17,59.44,71.72,92.14
NEET_Urban_Rural,1989,20.86,10.16,0.97,14.03,18.99,26.22,60.41
NEET_Youth_Modelled,1071,20.25,8.49,3.52,14.25,18.95,24.82,47.70
UR_Modelled,3213,8.75,7.23,0.97,4.02,6.50,10.68,50.39
UR_Urban_Rural,6457,7.82,6.67,0.17,3.28,5.78,9.86,51.47
Working_Poverty_Modelled,2560,6.29,8.41,0.01,0.92,3.71,7.18,43.77


---
### Siguiente etapa del flujo de trabajo
Con los datasets maestros generados y verificados en `data/processed/`, continúe en [`02_Auditoria_y_Validacion_2015_2025_DAX.ipynb`](02_Auditoria_y_Validacion_2015_2025_DAX.ipynb) para la auditoría de calidad (2015–2025) y la validación 1:1 de los 14 controles numéricos de `Dashboard_V2.pbix`.